# Objective 2 -- Productivity & Degradation: Plots

Reads the zonal-statistics and LandTrendr event-summary CSVs exported by
`productivity_degradation_landtrendr.ipynb` (downloaded from Drive into `outputs/tables/`) and
produces trend/degradation plots. No Earth Engine calls happen in this notebook -- it is a pure pandas/matplotlib/
seaborn post-processing step on already-exported tables.

Plots are deliberately consolidated rather than one-per-plan-line-item to keep the set focused
on the clearest productivity/degradation signal, and to complement rather than duplicate the
LandTrendr disturbance/recovery rasters already exported (this notebook covers site-level
*trend* and *summary* evidence; the rasters cover *where*).

In [23]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from eetools.visualization.plots import plot_single_metric_annually

import config

In [24]:
sns.set_theme(style="whitegrid")
config.PLOTS_DIR.mkdir(parents=True, exist_ok=True)

SITE_ORDER = [s["site_id"] for s in config.SITES]
SITE_LABELS = {s["site_id"]: s["site_name"] for s in config.SITES}
REFERENCE_SITE = "mbokishi"  # intact reference site
COMPARISON_SITES = [s for s in SITE_ORDER if s != REFERENCE_SITE]
PERIOD_ORDER = list(config.PERIODS.keys())  # baseline, pre, current

# Site polygon areas (ha) -- used to normalize Plot 6's disturbance/recovery hectares by site
# size; Mbokishi is ~4-7x larger than the other three sites, so raw hectare totals alone
# overstate its relative disturbance/recovery extent.
SITE_AREAS_HA = {
    site["site_id"]: gpd.read_file(site["path"]).to_crs(config.PROJECT_CRS).geometry.area.sum() / 10000
    for site in config.SITES
}

## Load tables

Landsat is used for every core trend/anomaly/boxplot panel below -- it's the only sensor with a
full 1984-2025 record, for headline trend maps and keeping the line-chart evidence directly comparable to the LandTrendr rasters (also built from Landsat). HLS/Sentinel-2 tables are available in `outputs/tables/` for future
sensor-specific or fine-scale follow-up.

In [25]:
landsat_ts = pd.read_csv(config.TABLES_DIR / "landsat_site_timeseries_annual_seasonal.csv")
landsat_period = pd.read_csv(config.TABLES_DIR / "landsat_site_summary_by_period.csv")
# Two independent LandTrendr runs (dry-season NBR, wet-season MSAVI2) -- see Plot 6.
landtrendr_nbrseg_summary = pd.read_csv(config.TABLES_DIR / "landtrendr_nbrseg_event_summary_by_site.csv")
landtrendr_msavi2seg_summary = pd.read_csv(config.TABLES_DIR / "landtrendr_msavi2seg_event_summary_by_site.csv")
# Per-site zonal stats of the LandTrendr fitted-trajectory stacks (both runs' self + FTV series) -- see Plot 8.
landtrendr_fitted_trajectory = pd.read_csv(config.TABLES_DIR / "landtrendr_fitted_trajectory_by_site.csv")
chirps_annual = pd.read_csv(config.TABLES_DIR / "chirps_annual_rainfall_1984_2025.csv")

## Plots 1-3: Core productivity / moisture / bare-ground trends

Dry-season Landsat, 1984-2025, one representative metric per interpretation-framework theme:
MSAVI2 for productivity/sparse-grassland condition, NDMI for moisture/woody
condition, BSI for bare-ground degradation pressure. Median rather than mean 
(less sensitive to outliers). Some 1980s-90s dry-season windows have zero cloud-free Landsat 5
scenes over this small AOI (a real data gap, not a bug - see the notebook's own empty-window
guard); those years show as a genuine break in the line rather than an interpolated value.

In [26]:
def plot_site_index_trend(df: pd.DataFrame, value_col: str, ylabel: str, title: str, filename: str) -> None:
    """Line plot of one Landsat dry-season index metric by site, 1984-2025."""
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.lineplot(
        data=df,
        x="year",
        y=value_col,
        hue="site_name",
        hue_order=[SITE_LABELS[s] for s in SITE_ORDER],
        marker="o",
        markersize=4,
        ax=ax,
    )
    ax.set_xlabel("Year")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(title="Site")
    fig.tight_layout()
    fig.savefig(config.PLOTS_DIR / filename, dpi=200, bbox_inches="tight")


landsat_dry = landsat_ts[landsat_ts["season"] == "dry"].copy()
landsat_wet = landsat_ts[landsat_ts["season"] == "wet"].copy()
landsat_dry["site_name"] = landsat_dry["site_id"].map(SITE_LABELS)
landsat_wet["site_name"] = landsat_wet["site_id"].map(SITE_LABELS)

### Plot 1: Productivity trend (MSAVI2)

In [ ]:
plot_site_index_trend(
    landsat_dry,
    "MSAVI2_median",
    "Median MSAVI2 (wet season)",
    "MSAVI2 Productivity trend by site , Landsat wet season (1984-2025)",
    "trend_msavi2_by_site.png",
)

### Plot 2: Moisture / woody-condition trend (NDMI)

In [ ]:
plot_site_index_trend(
    landsat_dry,
    "NDMI_median",
    "Median NDMI (dry season)",
    "NDMI Moisture / woody-condition trend by site, Landsat dry season (1984-2025)",
    "trend_ndmi_by_site.png",
)

### Plot 3: Bare-ground degradation-pressure trend (BSI)

In [ ]:
plot_site_index_trend(
    landsat_dry,
    "BSI_median",
    "Median BSI (dry season)",
    "BSI Bare-ground / degradation-pressure trend by site, Landsat dry season (1984-2025)",
    "trend_bsi_by_site.png",
)

## Plot 4: Reference-normalized MSAVI2 anomaly vs. Mbokishi

`site_relative_anomaly = site_metric - mbokishi_metric_same_year`- the
localized-degradation read: if a site tracks Mbokishi (near zero), the shared regional
rainfall/climate signal likely explains any shared trend; a site diverging from zero indicates
something happening locally rather than regionally. All three comparison sites combined on one
axis (rather than three near-identical separate charts) so their relative divergence is directly
comparable.

In [ ]:
def build_relative_anomaly(df: pd.DataFrame, value_col: str, reference_site: str) -> pd.DataFrame:
    """Long-format site_metric - reference_site_metric_same_year (plan Sec.10.2)."""
    pivot = df.pivot(index="year", columns="site_id", values=value_col)
    anomaly = pivot.drop(columns=reference_site).subtract(pivot[reference_site], axis=0)
    return anomaly.reset_index().melt(id_vars="year", var_name="site_id", value_name="anomaly")


msavi2_anomaly = build_relative_anomaly(landsat_dry, "MSAVI2_median", REFERENCE_SITE)
msavi2_anomaly["site_name"] = msavi2_anomaly["site_id"].map(SITE_LABELS)

fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(
    data=msavi2_anomaly,
    x="year",
    y="anomaly",
    hue="site_name",
    hue_order=[SITE_LABELS[s] for s in COMPARISON_SITES],
    marker="o",
    markersize=4,
    ax=ax,
)
ax.axhline(0, color="black", linewidth=1, linestyle="--")
ax.set_xlabel("Year")
ax.set_ylabel("MSAVI2 median, site minus Mbokishi")
ax.set_title("Reference-normalized MSAVI2 anomaly vs. Mbokishi, Landsat dry season (1984-2025)")
ax.legend(title="Site")
fig.tight_layout()
fig.savefig(config.PLOTS_DIR / "msavi2_anomaly_vs_mbokishi.png", dpi=200, bbox_inches="tight")

## Plot 5: Pre-establishment vs. current condition (boxplots)

Recent 2022-2025 vs. pre-establishment 2016-2021 condition per site, across the
same three metrics as Plots 1-3. The zonal-statistics export already computed each site's
per-period spatial distribution as percentiles (`_median`/`_p10`/`_p25`/`_p75`/`_p90`) rather
than exporting raw per-pixel values, so these boxes
are built directly from those percentiles via `Axes.bxp()`, median/p25/p75/p10/p90 map exactly
onto a boxplot's median/Q1/Q3/whiskers, giving a genuine per-site spatial distribution box
without needing raw pixel data.

In [ ]:
def build_boxplot_stats(df: pd.DataFrame, value_prefix: str) -> list[dict]:
    """Build matplotlib bxp() stats dicts directly from already-computed spatial percentiles."""
    stats = []
    for _, row in df.iterrows():
        stats.append(
            {
                "label": f"{SITE_LABELS[row['site_id']]}\n{row['period']}",
                "med": row[f"{value_prefix}_median"],
                "q1": row[f"{value_prefix}_p25"],
                "q3": row[f"{value_prefix}_p75"],
                "whislo": row[f"{value_prefix}_p10"],
                "whishi": row[f"{value_prefix}_p90"],
                "fliers": [],
            }
        )
    return stats


BOXPLOT_PERIODS = ["pre", "current"]
BOXPLOT_METRICS = [
    ("MSAVI2", "MSAVI2 (productivity)"),
    ("NDMI", "NDMI (moisture/woody)"),
    ("BSI", "BSI (bare ground)"),
]

period_subset = landsat_period[landsat_period["period"].isin(BOXPLOT_PERIODS)].copy()
period_subset["site_id"] = pd.Categorical(period_subset["site_id"], categories=SITE_ORDER, ordered=True)
period_subset["period"] = pd.Categorical(period_subset["period"], categories=BOXPLOT_PERIODS, ordered=True)
period_subset = period_subset.sort_values(["site_id", "period"])

fig, axes = plt.subplots(1, len(BOXPLOT_METRICS), figsize=(15, 5))
for ax, (metric, label) in zip(axes, BOXPLOT_METRICS):
    ax.bxp(build_boxplot_stats(period_subset, metric), showfliers=False)
    ax.set_title(label)
    ax.set_ylabel(metric)
    for tick_label in ax.get_xticklabels():
        tick_label.set_rotation(45)
        tick_label.set_horizontalalignment("right")

fig.suptitle("Pre-establishment (2016-2021) vs. current (2022-2025) condition by site, Landsat annual composites")
fig.tight_layout()
fig.savefig(config.PLOTS_DIR / "pre_vs_current_condition_boxplots.png", dpi=200, bbox_inches="tight")

## Plot 6: LandTrendr disturbance/recovery area by site (NBR run vs. MSAVI2 run)

Direct numeric complement to the LandTrendr disturbance/recovery rasters: the rasters show
*where* on the ground disturbance/recovery occurred, this shows the *aggregate extent* per site.
Normalized to **% of site area** rather than raw hectares -- Mbokishi is 4.4x Enarau, 7.3x
Corridor Phase 1, and 5.4x Corridor Phase 2 in area, so a raw-hectare comparison mostly reflects
site size rather than relative disturbance/recovery intensity (confirmed: once normalized, all
four sites cluster in a similar range rather than Mbokishi dominating). Rows compare the two
independent LandTrendr segmentations (dry-season NBR, wet-season MSAVI2) side by side rather than
summing them, since they come from separate segmentations of different indices and a combined
total wouldn't correspond to any single coherent measurement. Columns split the full 1984-2025
record from the "recent window" breakdown, whether disturbance is concentrated in 2016-2025 or
has accelerated further within 2022-2025. Y-axes are shared per column so the NBR-run and
MSAVI2-run rows are directly comparable in scale.

In [ ]:
CHANGE_TYPE_LABELS = {
    "disturbance_full": "Disturbance (1984-2025)",
    "recovery_full": "Recovery (1984-2025)",
    "disturbance_recent_2016_2025": "Disturbance (2016-2025)",
    "disturbance_recent_2022_2025": "Disturbance (2022-2025)",
    "recovery_recent_2016_2025": "Recovery (2016-2025)",
}
LANDTRENDR_RUNS = {
    "nbrseg": ("NBR (dry-season)", landtrendr_nbrseg_summary),
    "msavi2seg": ("MSAVI2 (wet-season)", landtrendr_msavi2seg_summary),
}
RECENT_CHANGE_TYPES = [
    "disturbance_recent_2016_2025",
    "disturbance_recent_2022_2025",
    "recovery_recent_2016_2025",
]
site_name_order = [SITE_LABELS[s] for s in SITE_ORDER]

fig, axes = plt.subplots(2, 2, figsize=(15, 11), sharey="col")

for row, (run_key, (run_label, run_df)) in enumerate(LANDTRENDR_RUNS.items()):
    run_df = run_df.copy()
    run_df["site_name"] = run_df["site_id"].map(SITE_LABELS)
    run_df["change_label"] = run_df["change_type"].map(CHANGE_TYPE_LABELS)
    run_df["pct_of_site_area"] = 100 * run_df["area_ha_sum"] / run_df["site_id"].map(SITE_AREAS_HA)

    full_record = run_df[run_df["change_type"].isin(["disturbance_full", "recovery_full"])]
    recent = run_df[run_df["change_type"].isin(RECENT_CHANGE_TYPES)]

    sns.barplot(
        data=full_record, x="site_name", y="pct_of_site_area", hue="change_label",
        order=site_name_order, ax=axes[row, 0],
    )
    axes[row, 0].set_title(f"{run_label} full record (1984-2025)")
    axes[row, 0].set_xlabel("")
    axes[row, 0].set_ylabel("% of site area")
    axes[row, 0].legend(title=None)
    axes[row, 0].tick_params(axis="x", rotation=20)

    sns.barplot(
        data=recent, x="site_name", y="pct_of_site_area", hue="change_label",
        order=site_name_order, ax=axes[row, 1],
    )
    axes[row, 1].set_title(f"{run_label} recent window (2016-2025)")
    axes[row, 1].set_xlabel("")
    axes[row, 1].set_ylabel("% of site area")
    axes[row, 1].legend(title=None)
    axes[row, 1].tick_params(axis="x", rotation=20)

fig.suptitle("LandTrendr disturbance/recovery normalized area by site, (NBR vs. MSAVI2)")
fig.tight_layout()
fig.savefig(config.PLOTS_DIR / "landtrendr_pct_area_by_site_nbr_vs_msavi2.png", dpi=200, bbox_inches="tight")

## Plot 7: Annual rainfall (CHIRPS)

Rainfall-confounding context: if Plots 1-4 show every site declining in the same
year, cross-check it against a rainfall dip here before reading it as localized degradation.

In [ ]:
fig, ax = plot_single_metric_annually(
    chirps_annual,
    title="Annual rainfall (CHIRPS), 1984-2025",
    date_col="date",
    metric_col="precipitation",
)
ax.set_ylabel("Annual precipitation (mm)")
fig.savefig(config.PLOTS_DIR / "chirps_annual_rainfall_1984_2025.png", dpi=200, bbox_inches="tight")

In [ ]:
chirps_annual_2000_2025 = chirps_annual[(chirps_annual.year >= 2000) & (chirps_annual.year <= 2025)].copy()

fig, ax = plot_single_metric_annually(
    chirps_annual_2000_2025,
    title="Annual rainfall (CHIRPS), 2000-2025",
    date_col="date",
    metric_col="precipitation",
)
ax.set_ylabel("Annual precipitation (mm)")
fig.savefig(config.PLOTS_DIR / "chirps_annual_rainfall_2000_2025.png", dpi=200, bbox_inches="tight")

## Plot 8: LandTrendr fitted-trajectory trend by site (self-fit)

The smoothed annual trajectory each LandTrendr run actually segmented on, plotted the same way
as Plots 1-3 -- lets a reviewer directly compare a run's raw yearly index (e.g. Plot 1's raw
MSAVI2) against the piecewise-linear fit `get_change_map`'s disturbance/recovery segments were
drawn from. Two panels, reusing `plot_site_index_trend`: the NBR run's own NBR fit (dry season)
and the MSAVI2 run's own MSAVI2 fit (wet season). The co-fitted FTV series (NDMI/MSAVI2/BSI for
the NBR run; NBR/NDMI/BSI for the MSAVI2 run) are also in `landtrendr_fitted_trajectory` (`run`/
`index` columns) for a future directional-agreement chart, not plotted individually here.

In [ ]:
nbrseg_self_fit = landtrendr_fitted_trajectory[
    (landtrendr_fitted_trajectory["run"] == "nbrseg") & (landtrendr_fitted_trajectory["index"] == "NBR")
].copy()

msavi2seg_self_fit = landtrendr_fitted_trajectory[
    (landtrendr_fitted_trajectory["run"] == "msavi2seg") & (landtrendr_fitted_trajectory["index"] == "MSAVI2")
].copy()

plot_site_index_trend(
    nbrseg_self_fit,
    "median",
    "Median fitted NBR (dry season, LandTrendr fit)",
    "LandTrendr fitted trajectory by site -- NBR run (dry-season NBR, 1984-2025)",
    "landtrendr_fitted_nbrseg_by_site.png",
)

plot_site_index_trend(
    msavi2seg_self_fit,
    "median",
    "Median fitted MSAVI2 (wet season, LandTrendr fit)",
    "LandTrendr fitted trajectory by site -- MSAVI2 run (wet-season MSAVI2, 1984-2025)",
    "landtrendr_fitted_msavi2seg_by_site.png",
)

## Notes & next steps

- The 1980s-90s dry-season data gaps visible in Plots 1-4 (breaks in the line) are real.
  some Landsat-5-only dry-season windows have zero cloud-free scenes over this ~89 km² AOI,
  not a plotting artifact. The same gaps can appear in Plot 8's fitted trajectories for the
  same reason.
- Plot 4 covers MSAVI2 only; the same `build_relative_anomaly` helper works unchanged for NDMI or
  BSI if a reviewer wants the moisture/bare-ground anomaly too.
- Plot 5's boxes are built from spatial percentiles already computed during the zonal-statistics
  export (median/p10/p25/p75/p90), not raw per-pixel values.
- Plot 6 compares the two independent LandTrendr runs (NBR, MSAVI2) side by side rather than
  merging them into one chart. Reading a site's MSAVI2-run event through its co-fitted
  NBR/NDMI/BSI FTV trajectory requires the per-pixel
  fitted-trajectory rasters, not this site-level area summary; this notebook doesn't (and can't,
  without a new zonal-statistics export) show that per-pixel directional-agreement check.
- Plot 8 only plots each run's own ("self") fitted index; `landtrendr_fitted_trajectory` also has
  the co-fitted FTV series (`run`/`index` columns -- e.g. `run="msavi2seg", index="NBR"`) for a
  future 4-line-per-site directional-agreement chart per the MSAVI2 interpretation framework, not
  built here yet.
- HLS (2015-present) and Sentinel-2 (2017-present, red-edge NDRE/CIred_edge) tables are already
  in `outputs/tables/` for a future fine-scale or recent-period follow-up.
- Cross-check any pattern seen here against the LandTrendr disturbance/recovery rasters and
  Objective 1's Dynamic World transitions before finalizing a reporting claim.